In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install gradio matplotlib seaborn scikit-learn tqdm -q

In [ ]:
import os, gzip, json, re, math, time, random, pickle
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm

#  Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

#  Directory Setup 
Path('results').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)
print('Directories ready.')

Using device: cuda
Directories ready.


---
## Dataset Loading, Preprocessing & Vocabulary Construction

In [ ]:

CATEGORIES = {
    'electronics': 'data/electronics.json.gz',
    'beauty':      'data/beauty.json.gz',
    'cellphones':  'data/cellphones.json.gz',
}

SAMPLES_PER_CATEGORY = 12000   
MAX_SEQ_LEN  = 128            
MAX_GEN_LEN  = 64             
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15



def load_category(path: str, category: str, max_samples: int) -> list[dict]:
    
    records = []
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            text   = obj.get('reviewText', '') or obj.get('summary', '')
            rating = obj.get('overall', None)
            if not text or rating is None:
                continue
            records.append({'text': text.strip(), 'rating': float(rating), 'category': category})
            if len(records) >= max_samples:
                break
    return records


all_records = []
for cat, path in CATEGORIES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing file: {path}  — place it in the data/ folder.')
    recs = load_category(path, cat, SAMPLES_PER_CATEGORY)
    print(f'  {cat:15s}: {len(recs):,} samples loaded')
    all_records.extend(recs)

random.shuffle(all_records)
print(f'\nTotal samples: {len(all_records):,}')

# Rating distribution
ratings = [r['rating'] for r in all_records]
for star in range(1, 6):
    print(f'  {star}★ : {ratings.count(float(star)):,}')

  electronics    : 12,000 samples loaded
  beauty         : 12,000 samples loaded
  cellphones     : 12,000 samples loaded

Total samples: 36,000
  1★ : 2,580
  2★ : 2,084
  3★ : 3,583
  4★ : 6,959
  5★ : 20,794


In [ ]:

# Label Engineering
#   Sentiment (3-class):  1-2 → 0 (Negative), 3 → 1 (Neutral), 4-5 → 2 (Positive)
#   Derived feature:  Review length class — Short (≤50 words) → 0, Medium → 1, Long (>150 words) → 2


def rating_to_sentiment(rating: float) -> int:
    if rating <= 2:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

def text_to_length_class(text: str) -> int:
    n = len(text.split())
    if n <= 50:
        return 0  # Short
    elif n <= 150:
        return 1  # Medium
    else:
        return 2  # Long

SENTIMENT_NAMES    = ['Negative', 'Neutral', 'Positive']
LENGTH_CLASS_NAMES = ['Short', 'Medium', 'Long']

for rec in all_records:
    rec['sentiment']    = rating_to_sentiment(rec['rating'])
    rec['length_class'] = text_to_length_class(rec['text'])

print('Label distribution:')
for i, name in enumerate(SENTIMENT_NAMES):
    cnt = sum(1 for r in all_records if r['sentiment'] == i)
    print(f'  Sentiment {name}: {cnt:,}')
for i, name in enumerate(LENGTH_CLASS_NAMES):
    cnt = sum(1 for r in all_records if r['length_class'] == i)
    print(f'  Length {name}: {cnt:,}')

Label distribution:
  Sentiment Negative: 4,664
  Sentiment Neutral: 3,583
  Sentiment Positive: 27,753
  Length Short: 15,666
  Length Medium: 13,615
  Length Long: 6,719


In [ ]:
# Text Cleaning & Simple Whitespace Tokenizer
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)          # remove HTML tags
    text = re.sub(r'https?://\S+', ' URL ', text)  # replace URLs
    text = re.sub(r"[^a-z0-9'\s]", ' ', text)     # keep letters, digits, apostrophes
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text: str) -> list[str]:
    """Simple whitespace tokenizer."""
    return clean_text(text).split()

# Test
sample = all_records[0]['text']
print('Original :', sample[:120])
print('Tokenized:', tokenize(sample)[:20])

Original : Sound quality is excellent for the price and it is very easy to use; I highly recommend this player.
Tokenized: ['sound', 'quality', 'is', 'excellent', 'for', 'the', 'price', 'and', 'it', 'is', 'very', 'easy', 'to', 'use', 'i', 'highly', 'recommend', 'this', 'player']


In [ ]:
# Train / Val / Test Split


n = len(all_records)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_records = all_records[:n_train]
val_records   = all_records[n_train : n_train + n_val]
test_records  = all_records[n_train + n_val:]

print(f'Train : {len(train_records):,}')
print(f'Val   : {len(val_records):,}')
print(f'Test  : {len(test_records):,}')

Train : 25,200
Val   : 5,400
Test  : 5,400


In [ ]:
# Vocabulary Construction 

MIN_FREQ = 3  
SPECIAL_TOKENS = ['<PAD>', '<UNK>', '<BOS>', '<EOS>']

counter = Counter()
for rec in tqdm(train_records, desc='Building vocab', disable=True):
    counter.update(tokenize(rec['text']))

vocab_tokens = SPECIAL_TOKENS + [tok for tok, cnt in counter.most_common() if cnt >= MIN_FREQ]
token2idx = {tok: i for i, tok in enumerate(vocab_tokens)}
idx2token = {i: tok for tok, i in token2idx.items()}

PAD_IDX = token2idx['<PAD>']
UNK_IDX = token2idx['<UNK>']
BOS_IDX = token2idx['<BOS>']
EOS_IDX = token2idx['<EOS>']
VOCAB_SIZE = len(vocab_tokens)

print(f'Vocabulary size: {VOCAB_SIZE:,}  (min_freq={MIN_FREQ})')

def encode(text: str, max_len: int, add_special: bool = False) -> list[int]:
    """Convert text to padded index list of length max_len."""
    toks = tokenize(text)
    ids  = [token2idx.get(t, UNK_IDX) for t in toks]
    if add_special:
        ids = [BOS_IDX] + ids + [EOS_IDX]
    ids = ids[:max_len]                         # truncate
    ids = ids + [PAD_IDX] * (max_len - len(ids))  # pad
    return ids

def decode(ids: list[int]) -> str:
    return ' '.join(idx2token.get(i, '<UNK>') for i in ids if i not in {PAD_IDX, BOS_IDX, EOS_IDX})

# Save vocab for later use
with open('results/vocab.pkl', 'wb') as f:
    pickle.dump({'token2idx': token2idx, 'idx2token': idx2token}, f)
print('Vocabulary saved.')

Vocabulary size: 16,443  (min_freq=3)
Vocabulary saved.


In [ ]:
# Reference Explanations for Decoder Training
SENTIMENT_LABELS = {0: 'negative', 1: 'neutral', 2: 'positive'}
LENGTH_LABELS    = {0: 'short', 1: 'medium-length', 2: 'long'}

def make_reference_explanation(rec: dict) -> str:
    sent = SENTIMENT_LABELS[rec['sentiment']]
    lc   = LENGTH_LABELS[rec['length_class']]
    cat  = rec['category']
    star = int(rec['rating'])
    templates = [
        f"this {lc} {cat} review expresses a {sent} sentiment with a {star} star rating",
        f"the reviewer gave {star} stars indicating {sent} feelings about this {cat} product",
        f"based on the {lc} review the customer has a {sent} opinion of this {cat} item",
    ]
    return random.choice(templates)

for rec in all_records:
    rec['explanation'] = make_reference_explanation(rec)

print('Sample explanation:', all_records[0]['explanation'])

Sample explanation: based on the short review the customer has a positive opinion of this electronics item
